# C2-linear-models — Practice p23

**Type:** constrained coding · **Difficulty:** core · **Concepts:** ols-rank-identifiability-and-pseudoinverse

Implement \`ols_pinv(X, y)\` for ordinary least squares at any rank.
Accept finite numeric \`X (n, p)\` and \`y (n,)\` with matching,
nonempty dimensions; raise \`ValueError\` before computing for malformed
or non-finite input. Do not mutate inputs.

Return float \`beta (p,)\` equal to
\[
\beta^+=X^+y,
\]
the minimum-Euclidean-norm least-squares coefficient. Its residual must
satisfy $X^T(X\beta^+-y)=0$.

**Required route:** exactly one \`np.linalg.pinv(X) @ y\` call per
accepted invocation and zero pinv calls on rejected input.

**Banned inside the function (zero points):** \`np.linalg.inv\`,
\`np.linalg.solve\`, \`np.linalg.lstsq\`, any spelling of
\`sklearn\` or \`statsmodels\`, loops, comprehensions, and recursion.
Do not hardcode visible examples; immutable full-rank and rank-deficient
secondary fixtures check coefficient, projection, residual, and
minimum-norm semantics at \`ATOL = 1e-10\`, \`RTOL = 0.0\`.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0


def ols_pinv(X, y):
    # YOUR CODE HERE
    ...

## Immutable contract check — do not edit

This checker recursively audits bytecode/source for banned shortcuts and
loops, instruments \`np.linalg.pinv\`, and independently evaluates
full-rank and deficient-rank fixtures. The deficient fixture checks the
shared projection/residual and verifies that nullspace shifts cannot
beat the pseudoinverse coefficient norm.

In [ ]:
import dis
import inspect
import types

_ORIGINAL_PINV_P23 = np.linalg.pinv


def _nested_code_p23(fn):
    pending = [fn.__code__]
    found = []
    while pending:
        current = pending.pop()
        found.append(current)
        pending.extend(
            item for item in current.co_consts
            if isinstance(item, types.CodeType)
        )
    return found


_codes_p23 = _nested_code_p23(ols_pinv)
_banned_names_p23 = {
    "inv", "solve", "lstsq", "sklearn", "statsmodels",
}
for _code_p23 in _codes_p23:
    _names_p23 = {name.lower() for name in _code_p23.co_names}
    assert not (_names_p23 & _banned_names_p23)
    assert ols_pinv.__name__ not in _code_p23.co_names
    _ops_p23 = {item.opname for item in dis.get_instructions(_code_p23)}
    assert "FOR_ITER" not in _ops_p23
    assert not any(name.startswith("JUMP_BACKWARD") for name in _ops_p23)

try:
    _source_p23 = inspect.getsource(ols_pinv).lower()
except (OSError, TypeError):
    _source_p23 = ""
assert all(token not in _source_p23 for token in (
    "np.linalg.inv(", "np.linalg.solve(", "np.linalg.lstsq(",
    "sklearn", "statsmodels",
))

_X_full_p23 = np.array([
    [1.0, -3.0],
    [1.0, -1.0],
    [1.0, 2.0],
    [1.0, 5.0],
    [1.0, 8.0],
])
_y_full_p23 = np.array([-2.0, 0.4, 3.7, 8.2, 11.5])
_X_def_p23 = np.array([
    [1.0, -2.0, -4.0],
    [1.0, -1.0, -2.0],
    [1.0, 1.0, 2.0],
    [1.0, 3.0, 6.0],
    [1.0, 6.0, 12.0],
])
_y_def_p23 = np.array([-1.0, 0.2, 2.5, 5.3, 9.1])
_z_def_p23 = np.array([0.0, -2.0, 1.0])
_fixtures_p23 = (
    (_X_full_p23, _y_full_p23, None),
    (_X_def_p23, _y_def_p23, _z_def_p23),
)

for _X_p23, _y_p23, _z_p23 in _fixtures_p23:
    _expected_p23 = _ORIGINAL_PINV_P23(_X_p23) @ _y_p23
    _X_before_p23 = _X_p23.copy()
    _y_before_p23 = _y_p23.copy()
    _pinv_calls_p23 = []

    def _counted_pinv_p23(a, *args, **kwargs):
        _pinv_calls_p23.append(np.array(a, copy=True))
        return _ORIGINAL_PINV_P23(a, *args, **kwargs)

    np.linalg.pinv = _counted_pinv_p23
    try:
        _beta_p23 = ols_pinv(_X_p23, _y_p23)
    finally:
        np.linalg.pinv = _ORIGINAL_PINV_P23

    assert len(_pinv_calls_p23) == 1
    assert isinstance(_beta_p23, np.ndarray)
    assert _beta_p23.shape == (_X_p23.shape[1],)
    assert np.issubdtype(_beta_p23.dtype, np.floating)
    assert np.isfinite(_beta_p23).all()
    assert np.array_equal(_X_p23, _X_before_p23)
    assert np.array_equal(_y_p23, _y_before_p23)
    assert np.allclose(_beta_p23, _expected_p23, atol=ATOL, rtol=RTOL)
    _pred_p23 = _X_p23 @ _beta_p23
    _resid_p23 = _pred_p23 - _y_p23
    _expected_resid_p23 = _X_p23 @ _expected_p23 - _y_p23
    assert np.allclose(
        _pred_p23, _X_p23 @ _expected_p23, atol=ATOL, rtol=RTOL,
    )
    assert np.allclose(
        _resid_p23, _expected_resid_p23, atol=ATOL, rtol=RTOL,
    )
    assert np.allclose(
        _X_p23.T @ _resid_p23,
        np.zeros(_X_p23.shape[1]),
        atol=ATOL, rtol=RTOL,
    )
    if _z_p23 is not None:
        assert np.allclose(
            _X_p23 @ _z_p23,
            np.zeros(_X_p23.shape[0]),
            atol=ATOL, rtol=RTOL,
        )
        for _scale_p23 in (-3.0, -0.5, 0.75, 4.0):
            _shifted_p23 = _beta_p23 + _scale_p23 * _z_p23
            assert np.allclose(
                _X_p23 @ _shifted_p23,
                _pred_p23,
                atol=ATOL, rtol=RTOL,
            )
            assert np.linalg.norm(_beta_p23) <= (
                np.linalg.norm(_shifted_p23) + ATOL
            )

_rejected_p23 = (
    (np.ones(3), np.ones(3)),
    (np.ones((3, 2)), np.ones((3, 1))),
    (np.ones((3, 2)), np.ones(2)),
    (np.empty((0, 2)), np.empty(0)),
    (np.array([[1.0, np.inf], [1.0, 2.0]]), np.ones(2)),
)
for _X_bad_p23, _y_bad_p23 in _rejected_p23:
    _pinv_calls_p23 = []

    def _reject_counted_pinv_p23(a, *args, **kwargs):
        _pinv_calls_p23.append(a)
        return _ORIGINAL_PINV_P23(a, *args, **kwargs)

    np.linalg.pinv = _reject_counted_pinv_p23
    try:
        try:
            ols_pinv(_X_bad_p23, _y_bad_p23)
        except ValueError:
            pass
        else:
            raise AssertionError("rejected input must raise ValueError")
    finally:
        np.linalg.pinv = _ORIGINAL_PINV_P23
    assert _pinv_calls_p23 == []